# 🤖 ai_text — Personal AI Workspace (Text Edition)

Successor to **ai_text_v1**. Major v2 changes:

- **PDF read + write** added — attach PDFs as input, generate PDFs as output.
- **Output Quality system removed** — the model writes whatever length your prompt requests. Say *"write a 2000-word essay"* or *"create a 3-page PDF"* or *"make an 8-slide deck"* and it targets that.
- **Tier picker remains** — controls which model runs (Fast / Balanced / Best / Custom).
- **Validation pass removed** (it was tied to the dropped Output Quality system).

Folder structure your PC needs:
```
<your project folder>/                ← run notebook from here
├── ai_text.ipynb                  ← THIS file
├── ai_text_files/                      ← Python package (all modules)
│   ├── __init__.py
│   ├── config.py    logger.py    installer.py
│   ├── conversation.py    file_readers.py    intent_router.py
│   ├── tier_manager.py    enhancement.py    model_installer.py
│   ├── builders_structure.py    builders_files.py    charts.py
│   ├── file_picker.py    ui.py
│   └── generated_files/           ← outputs land here
│       └── logs/                     ← log files land here
└── documents_v2/
    ├── ai_text_documentation.docx
    └── ai_text_setup_guide.docx
```

See `documents_v2/ai_text_setup_guide.docx` for full setup; `documents_v2/ai_text_documentation.docx` for what each module does.


In [1]:
# Cell 1 — detect GPU, then install dependencies
# Idempotent: skipped if already installed. May trigger a kernel restart
# the first time PyTorch is installed (re-run the cell after restart).
# v2: also installs pypdf (read PDFs) and reportlab (write PDFs).
# v3: checks for an NVIDIA GPU first — installs the CUDA build if found,
#     otherwise installs CPU-only packages and skips GPU-related setup.

import sys
import subprocess
from pathlib import Path

_NB_DIR = Path.cwd()
if str(_NB_DIR) not in sys.path:
    sys.path.insert(0, str(_NB_DIR))


def _detect_nvidia_gpu():
    """Return (has_gpu, info_lines). Uses nvidia-smi so we can decide
    before importing torch (which may not even be installed yet)."""
    try:
        out = subprocess.run(
            ["nvidia-smi",
             "--query-gpu=name,memory.total,driver_version",
             "--format=csv,noheader"],
            capture_output=True, text=True, timeout=10,
        )
    except (FileNotFoundError, subprocess.TimeoutExpired):
        return False, []
    if out.returncode != 0 or not out.stdout.strip():
        return False, []
    return True, [ln.strip() for ln in out.stdout.strip().splitlines() if ln.strip()]


has_gpu, gpu_lines = _detect_nvidia_gpu()
if has_gpu:
    print("\U0001F3A8 NVIDIA GPU detected:")
    for ln in gpu_lines:
        print(f"   • {ln}")
    print("   → installing PyTorch with CUDA support.")
else:
    print("\u2139\uFE0F  No NVIDIA GPU detected — running in CPU-only mode.")
    print("   → installing CPU-only packages; GPU/CUDA packages will be skipped.")

from ai_text_files.installer import ensure_dependencies
ensure_dependencies(use_gpu=has_gpu)

🎨 NVIDIA GPU detected:
   • NVIDIA RTX 2000 Ada Generation Laptop GPU, 8188 MiB, 591.86
   → installing PyTorch with CUDA support.
✅ PyTorch with CUDA already installed (torch 2.11.0+cu128, CUDA 12.8)
✅ All 14 dependencies already installed — skipping pip.
🎨 CUDA ready: NVIDIA RTX 2000 Ada Generation Laptop GPU (8.6 GB VRAM)


In [2]:
# Cell 2 — start the diagnostic logger, auto-pull missing Ollama models, load config
# Auto-pull: any model required by config.AUTO_PULL_TIERS that isn't
# already installed in Ollama will be pulled now. Default is to pull
# every tier (Fast + Balanced + Best), ~30 GB total, 25-45 min FIRST TIME.
# Subsequent runs are instant — they verify presence and skip downloads.
#
# To pull less, edit config.AUTO_PULL_TIERS in ai_text_files/config.py.
# To disable entirely, set config.AUTO_PULL_MISSING_MODELS = False.

from ai_text_files import config, logger, model_installer
logger.start_session()
model_installer.auto_pull_on_startup(on_line=print)

print(f"\nProject root  : {config.WORK_DIR}")
print(f"Output folder : {config.OUTPUT_DIR}")
print(f"Logs folder   : {config.LOGS_DIR}")
print(f"Log file      : {logger.LOG_FILE}")

✅ Config ready  |  C:\Clark\ramona\paper\PAMAP2
   Output → C:\Clark\ramona\paper\PAMAP2\ai_text_files\generated_files
   Active tier: balanced  ·  (Output Quality removed — specify length in your prompt)
   Best installed local: qwen3:14b  (8 model(s) found)
   `claude` CLI: C:\Users\HAhadyDolatsara\.local\bin\claude.EXE
📋 Diagnostic logger ready (v5.0) for ai_text
   Log file: C:\Clark\ramona\paper\PAMAP2\ai_text_files\generated_files\logs\ai_text_log_20260511_224905.txt
   (fresh file for this kernel session)
   Open the file at any time to see what's recorded.
────────────────────────────────────────────────────────────
🔄 Auto-pull on startup — checking 3 tier(s): fast, balanced, best
────────────────────────────────────────────────────────────
  'fast' tier — already complete, nothing to pull
  'balanced' tier — already complete, nothing to pull
  'best' tier — already complete, nothing to pull

────────────────────────────────────────────────────────────
✅ Auto-pull finished. Upd

In [3]:
# Cell 3 — render the UI
# This is the only widget you'll interact with for normal use.

from ai_text_files.ui import render_ui
render_ui()

HTML(value='\n        <h2 style=\'color:#1F3864;margin-bottom:2px\'>\n          🤖 ai_text — Personal AI Worksp…

Textarea(value='', description='📝 Topic / Question:', layout=Layout(height='130px', width='820px'), placeholde…

📋 logging active → C:\Clark\ramona\paper\PAMAP2\ai_text_files\generated_files\logs\ai_text_log_20260511_224905.txt


In [4]:
# Cell 4 (optional) — list everything in generated_files/

from ai_text_files.builders_files import list_outputs
list_outputs()

📁 C:\Clark\ramona\paper\PAMAP2\ai_text_files\generated_files

  Name                                                        Size
  ─────────────────────────────────────────────────────────────────
  Create_a_Python_3_script_that_simulates_20260511_223840.py     0.8 KB
  Write_a_self-contained_Python_script_tha_20260511_180851.py     1.4 KB
  Write_a_self-contained_Python_script_tha_20260511_223952.py     0.6 KB

📋 Logs in C:\Clark\ramona\paper\PAMAP2\ai_text_files\generated_files\logs:
  ai_text_log_20260511_223655.txt                            26.4 KB
  ai_text_log_20260511_224905.txt                            12.6 KB
  ai_text_v2_log_20260510_142256_pre_fixes.txt              244.7 KB
  ai_text_v2_log_20260511_180748.txt                         19.5 KB
